In [0]:
# Databricks notebook: proves the two implementations agree, row for row.
import sys, os
sys.path.append("/Workspace/Users/pronnoy1998@gmail.com/rearc-data-quest/src/alternatives/gold_pyspark_pipeline/transformations")
from gold_pyspark import population_stats, series_best_year, series_q01_with_population
from pyspark.testing import assertDataFrameEqual
obs = spark.table("rearc_quest.silver.silver_pr_observations")
dim = spark.table("rearc_quest.silver.silver_pr_series_dim")
pop = spark.table("rearc_quest.silver.silver_population")
pairs = {
    "gold_population_stats": population_stats(pop),
    "gold_series_best_year": series_best_year(obs, dim),
    "gold_prs30006032_q01_population": series_q01_with_population(obs, dim, pop),
    }
for name, py_df in pairs.items():
    sql_df = spark.table(f"rearc_quest.gold.{name}")
    sql_cols = sql_df.columns
    assertDataFrameEqual(py_df.select(sql_cols), sql_df, checkRowOrder=False)
    print(f"PARITY OK {name}: {sql_df.count()} rows")